In [1]:
!pip install transformers datasets
!pip install accelerate

In [3]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_chat.txt to cleaned_chat.txt


In [4]:
from datasets import Dataset
import pandas as pd

# Read and split into chunks
with open("cleaned_chat.txt", "r", encoding="utf-8") as f:
    data = f.read().strip().split("\n\n")

# Convert to dataset
df = pd.DataFrame(data, columns=["text"])
dataset = Dataset.from_pandas(df)

In [6]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer

model_name = "distilgpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # to avoid padding issues
model = GPT2LMHeadModel.from_pretrained(model_name)

# Tokenize data
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Map:   0%|          | 0/2664 [00:00<?, ? examples/s]

In [9]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

from transformers import TrainingArguments

# training_args = TrainingArguments(
#     output_dir="./mebot-model",
#     overwrite_output_dir=True,
#     per_device_train_batch_size=2,
#     num_train_epochs=3,
#     save_steps=500,
#     save_total_limit=1,
#     logging_steps=100,
#     fp16=False,
#     report_to="none",
# )


training_args = TrainingArguments(
    output_dir="./mebot-model",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    save_steps=10_000,
    save_total_limit=1,
    logging_steps=5,
    fp16=False,
    report_to="none",
)


# trainer = Trainer(
#     model=model,
#     args=training_args,
#     data_collator=data_collator,
#     train_dataset=tokenized_dataset,
# )


# Limit dataset for trial run
tokenized_dataset = tokenized_dataset.select(range(50))

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,5.458332
10,4.954020


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13, training_loss=5.210343507620005, metrics={'train_runtime': 98.5963, 'train_samples_per_second': 0.507, 'train_steps_per_second': 0.132, 'total_flos': 1633104691200.0, 'train_loss': 5.210343507620005, 'epoch': 1.0})

In [10]:
trainer.save_model("./mebot-model")
tokenizer.save_pretrained("./mebot-model")

# Zip the model for download
!zip -r mebot-model.zip ./mebot-model

# Download the zip
from google.colab import files
files.download("mebot-model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: mebot-model/ (stored 0%)
  adding: mebot-model/generation_config.json (deflated 25%)
  adding: mebot-model/tokenizer_config.json (deflated 48%)
  adding: mebot-model/tokenizer.json (deflated 82%)
  adding: mebot-model/checkpoint-13/ (stored 0%)
  adding: mebot-model/checkpoint-13/scheduler.pt (deflated 62%)
  adding: mebot-model/checkpoint-13/rng_state.pth (deflated 26%)
  adding: mebot-model/checkpoint-13/generation_config.json (deflated 25%)
  adding: mebot-model/checkpoint-13/tokenizer_config.json (deflated 48%)
  adding: mebot-model/checkpoint-13/tokenizer.json (deflated 82%)
  adding: mebot-model/checkpoint-13/optimizer.pt (deflated 8%)
  adding: mebot-model/checkpoint-13/model.safetensors (deflated 7%)
  adding: mebot-model/checkpoint-13/trainer_state.json (deflated 58%)
  adding: mebot-model/checkpoint-13/training_args.bin (deflated 53%)
  adding: mebot-model/checkpoint-13/config.json (deflated 53%)
  adding: mebot-model/model.safetensors (deflated 7%)
  adding: mebot-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer

# Load the trained model
model_path = "./mebot-model"
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path)

# Create a text generation pipeline
chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Try it with a prompt
prompt = "Friend: what's your plan for today?\nYou:"
output = chatbot(prompt, max_length=100, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

print(output[0]['generated_text'])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'pad_token_id', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Friend: what's your plan for today?
You: it's a couple of days ago but I am still in the process. I am still planning a day. I am not sure when I will be able to go to a meeting with the team, but I am going to talk to the team. I am not sure if I will have any idea to go to the meeting. I am not sure if it will be any more than the usual time, but the team has been there so far. I am sure I will have a lot of things to work on. I am still not sure what to do. What are you planning on?
You: I am going to be with the team. I am still planning a day. I am not sure if it will be any more than the usual time, but the team has been there so far. I am still planning a day. I am not sure if it will be any more than the usual time, but the team has been there so far. I am still planning a day. I am still planning a day. I am still planning a day.
You: I am still planning a day. I am still planning a day. I am still planning a day. I am still planning a day. I am still planning a day. I am stil

In [13]:
import re
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer

# Load the trained model
model_path = "./mebot-model"
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path)

# Create a text generation pipeline
chatbot = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Function to clean and format the bot's reply
def clean_reply(raw_output):
    # Extract reply after "You:"
    reply = raw_output.split("You:")[-1].split("Friend:")[0].strip()

    # Limit repeated emojis to 2 max
    reply = re.sub(
        r'([\U0001F600-\U0001F64F\U0001F300-\U0001F6FF\U0001F1E0-\U0001F1FF])\1{2,}',
        r'\1\1',
        reply
    )

    # Stop at first full stop or newline
    reply = re.split(r'[\n.!?]', reply)[0].strip()

    return reply


# Example prompt
prompt = "Friend:what's new?\nYou:"

# Generate a response
output = chatbot(
    prompt,
    max_length=50,
    num_return_sequences=1,
    pad_token_id=tokenizer.eos_token_id,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.2
)

raw = output[0]['generated_text']

# Clean the generated reply
reply = clean_reply(raw)

# Print the cleaned reply
print("You:", reply)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You: ha, my little friend
